# LeetCode 1518 — Water Bottles bằng Python

**Link đề:** https://leetcode.com/problems/water-bottles/
**Độ khó:** Easy · **Chủ đề:** Math, Simulation (mô phỏng)

```python
class Solution:
    def numWaterBottles(self, numBottles: int, numExchange: int) -> int:
        ...
```

### Mục tiêu học
1. Mô phỏng một quá trình theo **trạng thái** (số chai đầy, số vỏ rỗng) và vòng lặp `while`.
2. Dùng `//`, `%` và `divmod` để xử lý "đổi được bao nhiêu, còn dư bao nhiêu".
3. Viết 3 lời giải: mô phỏng từng chai → mô phỏng theo đợt → **công thức `O(1)`** `n + (n - 1) // (e - 1)`.
4. Hiểu vì sao công thức đúng (mỗi lần đổi "tốn ròng" `e - 1` vỏ), và các bẫy như "mượn chai".
5. Trace, test đối chiếu trên toàn bộ miền constraints, tự giải biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Ban đầu có `numBottles` chai nước **đầy**. Ở chợ, cứ `numExchange` **vỏ chai rỗng** đổi được **1 chai đầy**.

Uống một chai đầy thì nó thành một vỏ rỗng.

Trả về **số chai nước tối đa** có thể uống.

| numBottles | numExchange | Output | Giải thích |
|---:|---:|---:|---|
| 9 | 3 | 13 | uống 9 → 9 vỏ đổi 3 chai; uống 3 → 3 vỏ đổi 1; uống 1. Tổng `9 + 3 + 1` |
| 15 | 4 | 19 | uống 15 → đổi 3 (dư 3 vỏ); uống 3 → 6 vỏ đổi 1 (dư 2); uống 1 → 3 vỏ, không đủ đổi. Tổng `15 + 3 + 1` |

**Constraints:** `1 <= numBottles <= 100`, `2 <= numExchange <= 100`.

Chú ý: vỏ **dư** sau mỗi lần đổi **không mất đi** — nó được cộng dồn với vỏ mới (ví dụ 2: dư 3, cộng thêm 3 vỏ mới = 6).

## 2. Nền tảng Python cần biết

### 2.1. `//`, `%` và `divmod`

Có `empty` vỏ, đổi theo tỉ lệ `e`:
- số chai đầy đổi được: `empty // e` (chia lấy phần nguyên),
- số vỏ còn dư: `empty % e` (phần dư).

`divmod(a, b)` trả về **cả hai** cùng lúc dưới dạng tuple `(a // b, a % b)`.

In [1]:
empty, e = 15, 4
print("15 // 4 =", empty // e, "| 15 % 4 =", empty % e)
q, r = divmod(empty, e)
print("divmod(15, 4) =", (q, r))
print("kiểm tra: q * e + r =", q * e + r)

15 // 4 = 3 | 15 % 4 = 3
divmod(15, 4) = (3, 3)
kiểm tra: q * e + r = 15


### 2.2. Vòng lặp `while` — lặp khi điều kiện còn đúng

Khác `for` (biết trước số vòng), `while điều_kiện:` lặp cho tới khi điều kiện sai. Phải chắc chắn mỗi vòng làm điều kiện **tiến dần tới sai**, nếu không sẽ lặp vô hạn.

In [2]:
x = 20
steps = 0
while x >= 3:
    x -= 3
    steps += 1
print("trừ 3 được", steps, "lần, còn", x)

trừ 3 được 6 lần, còn 2


### 2.3. Gán nhiều biến cùng lúc (tuple assignment)

`a, b = b, a + b` tính **toàn bộ vế phải trước** rồi mới gán — tiện cập nhật nhiều biến trạng thái liên quan nhau mà không cần biến tạm.

In [3]:
full, empty = 0, 15
full, empty = empty // 4, empty % 4      # vế phải dùng giá trị CŨ của empty
print("full =", full, "| empty =", empty)

full = 3 | empty = 3


## 3. Cách 1 — Mô phỏng từng chai một (trực giác nhất)

Trạng thái: `full` (chai đầy), `empty` (vỏ rỗng), `drunk` (đã uống).

Lặp:
- Nếu còn chai đầy: uống 1 → `full -= 1`, `empty += 1`, `drunk += 1`.
- Ngược lại, nếu đủ vỏ: đổi → `empty -= e`, `full += 1`.
- Ngược lại: dừng.

In [4]:
def num_water_bottles_one_by_one(num_bottles: int, num_exchange: int) -> int:
    full, empty, drunk = num_bottles, 0, 0
    while True:
        if full > 0:                      # còn chai đầy -> uống 1 chai
            full -= 1
            empty += 1
            drunk += 1
        elif empty >= num_exchange:       # hết chai đầy nhưng đủ vỏ -> đổi 1 chai
            empty -= num_exchange
            full += 1
        else:                             # không còn gì để làm
            break
    return drunk

print(num_water_bottles_one_by_one(9, 3))    # 13
print(num_water_bottles_one_by_one(15, 4))   # 19

13
19


`while True:` lặp "mãi mãi", và ta thoát bằng `break` khi không còn hành động nào. Vòng lặp chắc chắn dừng vì mỗi lần đổi làm **giảm** tổng `full + empty` đi `e - 1 >= 1`, còn mỗi lần uống giữ nguyên tổng nhưng giảm `full`.

## 4. Nút thắt — mỗi lần chỉ xử lý 1 chai

Cách 1 đúng nhưng số vòng lặp ≈ số chai uống được + số lần đổi. Thực tế ta có thể **uống hết một lượt** và **đổi hết một lượt**: có `empty` vỏ thì đổi ngay `empty // e` chai, dư `empty % e`.

In [5]:
def count_loops_one_by_one(n, e):
    full, empty, loops = n, 0, 0
    while True:
        loops += 1
        if full > 0:
            full -= 1; empty += 1
        elif empty >= e:
            empty -= e; full += 1
        else:
            break
    return loops

print("cách 1 với (100, 2):", count_loops_one_by_one(100, 2), "vòng lặp")

cách 1 với (100, 2): 299 vòng lặp


## 5. Cách 2 — Mô phỏng theo đợt với `divmod` (lời giải chính)

Ý tưởng:
1. Uống hết `numBottles` chai ban đầu: `drunk = n`, `empty = n`.
2. Khi `empty >= e`:
   - `new_full, empty = divmod(empty, e)` → đổi được `new_full` chai, dư `empty` vỏ.
   - Uống ngay `new_full` chai: `drunk += new_full`, và chúng thành vỏ: `empty += new_full`.
3. Hết đủ vỏ → trả `drunk`.

In [6]:
def num_water_bottles_batch(num_bottles: int, num_exchange: int) -> int:
    drunk = num_bottles               # uống hết số chai ban đầu
    empty = num_bottles               # ... thành bấy nhiêu vỏ
    while empty >= num_exchange:
        new_full, empty = divmod(empty, num_exchange)   # đổi: được new_full chai, dư empty vỏ
        drunk += new_full                               # uống ngay các chai mới
        empty += new_full                               # chúng thành vỏ, cộng với vỏ dư
    return drunk

print(num_water_bottles_batch(9, 3))    # 13
print(num_water_bottles_batch(15, 4))   # 19

13
19


## 6. TRACE từng bước — cách 2

Ví dụ `numBottles = 15, numExchange = 4`:

| đợt | empty trước | `divmod(empty, 4)` | uống thêm | drunk | empty sau = dư + mới |
|---:|---:|---|---:|---:|---:|
| khởi đầu | – | – | 15 | 15 | 15 |
| 1 | 15 | (3, 3) | 3 | 18 | 3 + 3 = 6 |
| 2 | 6 | (1, 2) | 1 | 19 | 2 + 1 = 3 |
| dừng | 3 | 3 < 4 | – | **19** | 3 |

In [7]:
def trace_water_bottles(n, e):
    drunk, empty = n, n
    print(f"Uống {n} chai ban đầu -> drunk = {drunk}, empty = {empty}")
    round_no = 1
    while empty >= e:
        new_full, rest = divmod(empty, e)
        drunk += new_full
        print(f"Đợt {round_no}: {empty} vỏ -> đổi {new_full} chai (dư {rest} vỏ); uống -> drunk = {drunk}; vỏ = {rest} + {new_full} = {rest + new_full}")
        empty = rest + new_full
        round_no += 1
    print(f"Dừng: còn {empty} vỏ < {e}. Đáp án = {drunk}")
    return drunk

trace_water_bottles(15, 4)
print()
trace_water_bottles(9, 3)

Uống 15 chai ban đầu -> drunk = 15, empty = 15
Đợt 1: 15 vỏ -> đổi 3 chai (dư 3 vỏ); uống -> drunk = 18; vỏ = 3 + 3 = 6
Đợt 2: 6 vỏ -> đổi 1 chai (dư 2 vỏ); uống -> drunk = 19; vỏ = 2 + 1 = 3
Dừng: còn 3 vỏ < 4. Đáp án = 19

Uống 9 chai ban đầu -> drunk = 9, empty = 9
Đợt 1: 9 vỏ -> đổi 3 chai (dư 0 vỏ); uống -> drunk = 12; vỏ = 0 + 3 = 3
Đợt 2: 3 vỏ -> đổi 1 chai (dư 0 vỏ); uống -> drunk = 13; vỏ = 0 + 1 = 1
Dừng: còn 1 vỏ < 3. Đáp án = 13


13

## 7. Cách 3 — Công thức `O(1)`

Nhìn lại một lần đổi: đưa `e` vỏ, nhận 1 chai đầy, uống xong lại có 1 vỏ. **Ròng** (net), mỗi lần đổi số vỏ giảm `e - 1`, và ta uống thêm đúng 1 chai.

Sau khi uống `n` chai ban đầu có `n` vỏ. Ta còn đổi được khi số vỏ `>= e`. Câu hỏi: đổi được tối đa bao nhiêu lần `k`?

- Sau `k` lần đổi, số vỏ là `n - k(e - 1)`.
- Lần đổi thứ `k` thực hiện được khi **trước nó** số vỏ `n - (k-1)(e-1) >= e`, tức `n - k(e-1) >= 1`, tức `k(e - 1) <= n - 1`.
- Vậy `k_max = (n - 1) // (e - 1)`.

```
đáp án = n + (n - 1) // (e - 1)
```

Kiểm tra: `(9, 3)`: `9 + 8 // 2 = 13` ✔. `(15, 4)`: `15 + 14 // 3 = 15 + 4 = 19` ✔.

Trực giác cho điều kiện `>= 1`: luôn phải **giữ lại ít nhất 1 vỏ** ở cuối — vỏ của chai cuối cùng ta uống không bao giờ được "dùng ròng".

In [8]:
def num_water_bottles_formula(num_bottles: int, num_exchange: int) -> int:
    return num_bottles + (num_bottles - 1) // (num_exchange - 1)

for n, e in [(9, 3), (15, 4), (1, 2), (5, 5), (100, 2)]:
    print(f"n={n:>3}, e={e:>3}: công thức = {num_water_bottles_formula(n, e):>3} | mô phỏng = {num_water_bottles_batch(n, e)}")

n=  9, e=  3: công thức =  13 | mô phỏng = 13
n= 15, e=  4: công thức =  19 | mô phỏng = 19
n=  1, e=  2: công thức =   1 | mô phỏng = 1
n=  5, e=  5: công thức =   6 | mô phỏng = 6
n=100, e=  2: công thức = 199 | mô phỏng = 199


### Trace công thức: số vỏ sau mỗi lần đổi (net)

Với `(15, 4)`: vỏ `15 → 12 → 9 → 6 → 3`, mỗi bước giảm `e - 1 = 3`; dừng khi còn `3 < 4`. Có 4 lần đổi → uống `15 + 4 = 19`. (Ở cách 2, đợt 1 gom 3 lần đổi cùng lúc, đợt 2 là lần thứ 4.)

In [9]:
def trace_net(n, e):
    empty, k = n, 0
    seq = [empty]
    while empty >= e:
        empty -= (e - 1)        # đưa e vỏ, nhận 1 chai, uống xong thêm 1 vỏ
        k += 1
        seq.append(empty)
    print(f"(n={n}, e={e}) vỏ: {' -> '.join(map(str, seq))} | số lần đổi k = {k} = (n-1)//(e-1) = {(n - 1) // (e - 1)}")

trace_net(15, 4)
trace_net(9, 3)
trace_net(10, 10)

(n=15, e=4) vỏ: 15 -> 12 -> 9 -> 6 -> 3 | số lần đổi k = 4 = (n-1)//(e-1) = 4
(n=9, e=3) vỏ: 9 -> 7 -> 5 -> 3 -> 1 | số lần đổi k = 4 = (n-1)//(e-1) = 4
(n=10, e=10) vỏ: 10 -> 1 | số lần đổi k = 1 = (n-1)//(e-1) = 1


## 8. Vì sao đúng?

**Cách 2 — invariant:** đầu mỗi vòng `while`, `drunk` = tổng số chai đã uống, `empty` = số vỏ đang có, và **không còn chai đầy nào** (vì ta uống ngay sau khi đổi).
- Đổi tối đa có thể: `empty // e` chai; giữ lại `empty % e` vỏ. Đổi ít hơn cũng không có lợi: vỏ giữ lại vẫn còn đó cho vòng sau, kết quả cuối không đổi — tổng số lần đổi chỉ phụ thuộc vào việc còn `>= e` vỏ hay không.
- Vòng dừng khi `empty < e`: không thể đổi nữa, không còn chai đầy → `drunk` là tối đa.
- Dừng hữu hạn: mỗi vòng `empty` giảm `new_full * (e - 1) >= 1`.

**Cách 3:** chứng minh ở mục 7 — số vỏ sau `k` lần đổi là `n - k(e - 1)`, thực hiện được lần thứ `k` ⇔ `k(e-1) <= n-1`.

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def numWaterBottles(self, numBottles: int, numExchange: int) -> int:
        drunk = empty = numBottles
        while empty >= numExchange:
            new_full, empty = divmod(empty, numExchange)
            drunk += new_full
            empty += new_full
        return drunk

sol = Solution()
assert sol.numWaterBottles(9, 3) == 13
assert sol.numWaterBottles(15, 4) == 19
print("PASS ví dụ đề")

PASS ví dụ đề


Bản công thức (cũng Accepted):
```python
class Solution:
    def numWaterBottles(self, numBottles: int, numExchange: int) -> int:
        return numBottles + (numBottles - 1) // (numExchange - 1)
```
`numExchange >= 2` nên `numExchange - 1 >= 1`, không bao giờ chia cho 0.

## 10. Phân tích Time / Space

Gọi `n = numBottles`, `e = numExchange`.

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Từng chai | `O(n)` (≈ số chai uống + số lần đổi, tối đa ~`2n`) | `O(1)` | dễ hình dung |
| 2. Theo đợt | `O(log_e n)` | `O(1)` | mỗi đợt số vỏ giảm khoảng `e` lần |
| 3. Công thức | `O(1)` | `O(1)` | cần lập luận toán |

Vì sao cách 2 là `O(log_e n)`? Sau mỗi đợt, số vỏ mới ≈ `empty / e + empty % e < empty / e + e`, tức giảm khoảng `e` lần mỗi đợt (cho tới khi còn cỡ `e`).

In [11]:
def count_rounds(n, e):
    empty, rounds = n, 0
    while empty >= e:
        q, r = divmod(empty, e)
        empty = q + r
        rounds += 1
    return rounds

for n, e in [(100, 2), (100, 10), (10**9, 2), (10**9, 10)]:
    print(f"n={n:>10}, e={e:>2}: số đợt cách 2 = {count_rounds(n, e)}")

n=       100, e= 2: số đợt cách 2 = 7
n=       100, e=10: số đợt cách 2 = 2
n=1000000000, e= 2: số đợt cách 2 = 30
n=1000000000, e=10: số đợt cách 2 = 9


(Với `e = 2`, mỗi đợt gần như chỉ đổi được ít khi `empty` nhỏ — nhưng tổng số đợt vẫn chỉ cỡ `log₂ n` cộng vài vòng cuối.)

## 11. Kiểm thử: mẫu đề, biên, đối chiếu toàn miền

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<30} -> {actual}")

tests = [
    ("đề 1",                         9, 3, 13),
    ("đề 2 (có vỏ dư)",              15, 4, 19),
    ("1 chai, không đổi được",       1, 2, 1),
    ("n < e",                        3, 5, 3),
    ("n == e: đổi đúng 1 lần",       5, 5, 6),
    ("e = 2 (đổi nhiều nhất)",       100, 2, 199),
    ("biên trên",                    100, 100, 101),
    ("2 chai, e = 2",                2, 2, 3),
]
for name, n, e, exp in tests:
    for f in (num_water_bottles_one_by_one, num_water_bottles_batch, num_water_bottles_formula):
        assert f(n, e) == exp, (f.__name__, n, e)
    check(name, Solution().numWaterBottles(n, e), exp)

PASS  đề 1                           -> 13
PASS  đề 2 (có vỏ dư)                -> 19
PASS  1 chai, không đổi được         -> 1
PASS  n < e                          -> 3
PASS  n == e: đổi đúng 1 lần         -> 6
PASS  e = 2 (đổi nhiều nhất)         -> 199
PASS  biên trên                      -> 101
PASS  2 chai, e = 2                  -> 3


In [13]:
# Đối chiếu TOÀN BỘ miền constraints: 100 x 99 cặp
cnt = 0
for n in range(1, 101):
    for e in range(2, 101):
        a = num_water_bottles_one_by_one(n, e)
        assert num_water_bottles_batch(n, e) == a
        assert num_water_bottles_formula(n, e) == a
        cnt += 1
print(f"Đã đối chiếu {cnt} cặp (n, e): 3 cách khớp nhau. PASS")

Đã đối chiếu 9900 cặp (n, e): 3 cách khớp nhau. PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Bỏ mất vỏ dư
```python
new_full = empty // e
empty = new_full          # SAI: quên cộng empty % e
```
Với `(15, 4)`: đợt 1 đổi 3 nhưng vứt 3 vỏ dư → chỉ còn 3 vỏ → dừng → `18` thay vì `19`.

### Lỗi 2: Chỉ đổi một lần (quên vòng lặp)
```python
return n + n // e         # SAI: chai mới đổi về cũng tạo vỏ để đổi tiếp
```

### Lỗi 3: Quên rằng chai đổi về cũng thành vỏ
```python
new_full, empty = divmod(empty, e)
drunk += new_full         # SAI nếu thiếu dòng empty += new_full
```

### Lỗi 4: Công thức nhầm `n + n // (e - 1)` — "mượn" vỏ cuối
Ví dụ `(4, 2)`: đúng là `4 + 3 // 1 = 7`, nhưng `4 + 4 // 1 = 8` — tức đã "mượn" một chai rồi trả bằng vỏ, điều đề **không** cho phép.

In [14]:
def wrong_lose_rest(n, e):
    drunk = empty = n
    while empty >= e:
        new_full = empty // e
        drunk += new_full
        empty = new_full                  # lỗi 1
    return drunk

def wrong_once(n, e):
    return n + n // e                     # lỗi 2

def wrong_no_new_empty(n, e):
    drunk = empty = n
    while empty >= e:
        new_full, empty = divmod(empty, e)
        drunk += new_full                 # lỗi 3: thiếu empty += new_full
    return drunk

def wrong_formula(n, e):
    return n + n // (e - 1)               # lỗi 4

for n, e in [(15, 4), (9, 3), (4, 2)]:
    print(f"({n},{e}) đúng={num_water_bottles_formula(n, e)} | lỗi1={wrong_lose_rest(n, e)} | lỗi2={wrong_once(n, e)} | lỗi3={wrong_no_new_empty(n, e)} | lỗi4={wrong_formula(n, e)}")

(15,4) đúng=19 | lỗi1=18 | lỗi2=18 | lỗi3=18 | lỗi4=20
(9,3) đúng=13 | lỗi1=13 | lỗi2=12 | lỗi3=12 | lỗi4=13
(4,2) đúng=7 | lỗi1=7 | lỗi2=6 | lỗi3=6 | lỗi4=8


## 13. Mẹo nhận dạng pattern

- "Đổi `k` cái cũ lấy 1 cái mới, cái mới dùng xong lại thành cái cũ" → **mô phỏng theo đợt với `divmod`**, nhớ cộng phần dư.
- Muốn `O(1)`: tìm **lượng giảm ròng mỗi lần** (ở đây `e - 1`) và **điều kiện dừng** (`>= e` trước khi đổi ⇔ còn `>= 1` sau khi đổi).
- Quá trình có trạng thái nhỏ (vài biến) → vẽ bảng trạng thái theo đợt trước khi code.

Họ bài: 3100 Water Bottles II (tỉ lệ đổi tăng dần sau mỗi lần), các bài "đổi vỏ kẹo / vỏ chai" kinh điển trong sách nhập môn.

## 14. Biến thể

### Biến thể A — Water Bottles II (LeetCode 3100)
Mỗi lần đổi xong, `numExchange` **tăng thêm 1**, và mỗi lần chỉ được đổi **1 chai**. Không còn công thức đơn giản → mô phỏng.

In [15]:
def max_bottles_drunk_ii(num_bottles, num_exchange):
    drunk = empty = num_bottles
    while empty >= num_exchange:
        empty -= num_exchange
        num_exchange += 1
        drunk += 1
        empty += 1
    return drunk

print(max_bottles_drunk_ii(13, 6))   # 15
print(max_bottles_drunk_ii(10, 3))   # 13

15
13


### Biến thể B — Được phép "mượn" 1 chai
Nếu cuối cùng còn `e - 1` vỏ, có thể mượn 1 chai đầy, uống, rồi trả `e` vỏ. Khi đó đáp án đúng là `n + n // (e - 1)` — chính là "công thức sai" ở lỗi 4!

In [16]:
def with_borrow(n, e):
    drunk = num_water_bottles_formula(n, e)
    # số vỏ còn lại sau quá trình chuẩn:
    leftover = n - ((n - 1) // (e - 1)) * (e - 1)
    if leftover == e - 1:
        drunk += 1                     # mượn 1 chai, uống, trả e vỏ
    return drunk

for n, e in [(4, 2), (15, 4), (9, 3), (10, 4)]:
    print(f"({n},{e}) mượn: {with_borrow(n, e)} | n + n//(e-1) = {n + n // (e - 1)}")

(4,2) mượn: 8 | n + n//(e-1) = 8
(15,4) mượn: 20 | n + n//(e-1) = 20
(9,3) mượn: 13 | n + n//(e-1) = 13
(10,4) mượn: 13 | n + n//(e-1) = 13


## 15. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (tính tay)
Lập bảng đợt cho `numBottles = 20, numExchange = 6`. Trả về đáp án.

In [17]:
def exercise_by_hand() -> int:
    # TODO: uống 20 -> 20 vỏ -> divmod(20, 6) = ? -> ...
    pass

# assert exercise_by_hand() == num_water_bottles_batch(20, 6)

### Bài 2 (điền chỗ trống — theo đợt)

In [18]:
def exercise_water_bottles(num_bottles: int, num_exchange: int) -> int:
    drunk = empty = num_bottles
    while empty >= num_exchange:
        # TODO 1: new_full, empty = ?
        # TODO 2: cập nhật drunk
        # TODO 3: cập nhật empty
        pass
    return drunk

# assert exercise_water_bottles(9, 3) == 13
# assert exercise_water_bottles(15, 4) == 19

### Bài 3 (số vỏ còn lại cuối cùng)
Trả về số vỏ rỗng còn lại khi không thể đổi nữa.

In [19]:
def exercise_leftover_empty(num_bottles: int, num_exchange: int) -> int:
    # TODO
    pass

# assert exercise_leftover_empty(15, 4) == 3
# assert exercise_leftover_empty(9, 3) == 1

### Bài 4 (đổi ra nhiều chai)
Giờ cứ `e` vỏ đổi được `g` chai đầy (`1 <= g < e`). Viết mô phỏng theo đợt.

Ví dụ `n = 10, e = 3, g = 2` → `drunk = 26`.

In [20]:
def exercise_exchange_many(n: int, e: int, g: int) -> int:
    # TODO
    pass

# assert exercise_exchange_many(10, 3, 2) == 26
# assert exercise_exchange_many(9, 3, 1) == 13

### Bài 5 (bài toán ngược)
Muốn uống **ít nhất** `target` chai với tỉ lệ đổi `e`, cần mua ít nhất bao nhiêu chai ban đầu? (Dùng vòng lặp tăng dần hoặc suy từ công thức.)

In [21]:
def exercise_min_initial(target: int, e: int) -> int:
    # TODO
    pass

# assert exercise_min_initial(13, 3) == 9
# assert exercise_min_initial(19, 4) == 15
# assert exercise_min_initial(1, 2) == 1

### Bài 6 (mini challenge — vỏ và nắp)
Mỗi chai sau khi uống cho **1 vỏ và 1 nắp**. Chợ cho đổi: `a` vỏ → 1 chai, **hoặc** `b` nắp → 1 chai. Tính số chai tối đa uống được (mô phỏng: cứ đổi khi còn đổi được).

Quy tắc mô phỏng: mỗi đợt đổi **hết** những gì đổi được theo cả hai kiểu, uống ngay, rồi lặp lại. Ví dụ `n = 10, a = 2, b = 4` → `35`.

In [22]:
def exercise_bottles_and_caps(n: int, a: int, b: int) -> int:
    # TODO: trạng thái (empty, caps); mỗi chai mới uống cộng 1 vỏ + 1 nắp
    pass

# assert exercise_bottles_and_caps(10, 2, 4) == 35
# assert exercise_bottles_and_caps(9, 3, 10**9) == 13   # nắp vô dụng -> giống bài gốc

## 16. Gợi ý ba mức

**Bài 1** — M1: bảng giống mục 6. M2: `divmod(20, 6) = (3, 2)` → vỏ `2 + 3 = 5`. M3: `5 < 6` → dừng, đáp án `20 + 3`.

**Bài 2** — M1: đổi – uống – cộng vỏ. M2: `new_full, empty = divmod(empty, num_exchange)`. M3: `drunk += new_full; empty += new_full`.

**Bài 3** — M1: chạy mô phỏng và trả `empty` thay cho `drunk`. M2: hoặc công thức `n - k(e - 1)` với `k = (n-1)//(e-1)`. M3: kiểm tra `(15,4)`: `15 - 4*3 = 3`.

**Bài 4** — M1: mỗi đợt đổi được `(empty // e) * g` chai. M2: `q, r = divmod(empty, e)`; `new_full = q * g`. M3: `drunk += new_full; empty = r + new_full`.

**Bài 5** — M1: hàm "số chai uống được" tăng theo `n`. M2: thử `n = 1, 2, ...` tới khi `num_water_bottles_formula(n, e) >= target`. M3: `n` tối đa là `target` nên vòng lặp dừng.

**Bài 6** — M1: lặp khi `empty >= a or caps >= b`. M2: mỗi vòng đổi hết theo cả hai kiểu: `q1, empty = divmod(empty, a)`, `q2, caps = divmod(caps, b)`. M3: `new = q1 + q2; drunk += new; empty += new; caps += new`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [23]:
def answer_by_hand():
    return 23

def answer_water_bottles(num_bottles, num_exchange):
    drunk = empty = num_bottles
    while empty >= num_exchange:
        new_full, empty = divmod(empty, num_exchange)
        drunk += new_full
        empty += new_full
    return drunk

def answer_leftover_empty(num_bottles, num_exchange):
    empty = num_bottles
    while empty >= num_exchange:
        q, r = divmod(empty, num_exchange)
        empty = q + r
    return empty

def answer_exchange_many(n, e, g):
    drunk = empty = n
    while empty >= e:
        q, r = divmod(empty, e)
        new_full = q * g
        drunk += new_full
        empty = r + new_full
    return drunk

def answer_min_initial(target, e):
    n = 1
    while num_water_bottles_formula(n, e) < target:
        n += 1
    return n

def answer_bottles_and_caps(n, a, b):
    drunk = empty = caps = n
    while empty >= a or caps >= b:
        q1, empty = divmod(empty, a)
        q2, caps = divmod(caps, b)
        new = q1 + q2
        drunk += new
        empty += new
        caps += new
    return drunk

assert answer_by_hand() == num_water_bottles_batch(20, 6) == 23
assert answer_water_bottles(9, 3) == 13 and answer_water_bottles(15, 4) == 19
assert answer_leftover_empty(15, 4) == 3 and answer_leftover_empty(9, 3) == 1
for n in range(1, 101):
    for e in range(2, 30):
        assert answer_leftover_empty(n, e) == n - ((n - 1) // (e - 1)) * (e - 1)
assert answer_exchange_many(10, 3, 2) == 26
assert answer_exchange_many(9, 3, 1) == 13
assert answer_min_initial(13, 3) == 9 and answer_min_initial(19, 4) == 15 and answer_min_initial(1, 2) == 1
assert answer_bottles_and_caps(10, 2, 4) == 35
assert answer_bottles_and_caps(9, 3, 10**9) == 13
assert answer_bottles_and_caps(10, 2, 10**9) == num_water_bottles_formula(10, 2)   # nắp vô dụng -> quay về bài gốc
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numWaterBottles(self, numBottles: int, numExchange: int) -> int:
        drunk = empty = numBottles                    # uống hết chai ban đầu
        while empty >= numExchange:
            new_full, empty = divmod(empty, numExchange)   # đổi, giữ phần dư
            drunk += new_full                          # uống chai mới
            empty += new_full                          # chai mới thành vỏ
        return drunk
```

Công thức: `numBottles + (numBottles - 1) // (numExchange - 1)`.

**Checklist trước khi submit:**
- [ ] Giữ lại **vỏ dư** `empty % e`?
- [ ] Chai đổi về cũng **thành vỏ** sau khi uống?
- [ ] Lặp tới khi `empty < e` (không chỉ đổi một lần)?
- [ ] Công thức dùng `n - 1` (không "mượn" chai)?
- [ ] Đã test `n < e`, `n == e`, `e = 2`?